In [1]:
# Deterministic Optimization (ISYE 6669)
# Minimizing Energy Costs for a Large-Scale Manufacturing

#Importing libraries
import numpy as np
import cvxpy as cp
import csv

In [2]:
#Loading datasets from CSV 
dataset = np.genfromtxt("6669_Project_Electricity_Demand.csv", delimiter=',')
e_demand = dataset[1:73, 1]
dataset = np.genfromtxt("6669_Project_Electricity_Prices.csv", delimiter=',')
e_prices = dataset[1:73, 2]
dataset = np.genfromtxt("6669_Project_Hydrogen_Demand.csv", delimiter=',')
h_demand = dataset[1:73, 1]
dataset = np.genfromtxt("6669_Project_Solar_Forecast.csv", delimiter=',')
solar = dataset[1:73, 1]

FileNotFoundError: 6669_Project_Electricity_Demand.csv not found.

In [6]:
# Defining the number of hours/time range
num_of_hours = 72
hour_numbers = list(range(0, num_of_hours))

# Defining decision variables
H_variables = cp.Variable(72)
E_variables = cp.Variable(72)
H_storage_variables = cp.Variable(73)  
# 72 hours

# Defining the new demand after Solar has been fully utilised 
# available solar energy.
e_new_demand = []
for i in range(0, num_of_hours):
    e_new_demand.append(e_demand[i] - 2 * solar[i])
    if e_new_demand[i] < 0:
        e_new_demand[i] = 0

In [7]:
# Create constraints using a for loop
constraints = []
for i in range(0, num_of_hours):
    constraints.append(H_variables[i] + H_storage_variables[i] >= h_demand[i])
    constraints.append(E_variables[i] == e_new_demand[i])
    constraints.append(H_storage_variables[i] - (h_demand[i] - H_variables[i]) - H_storage_variables[i + 1] == 0)

# Sign Constraints
    constraints.append(H_variables[i] >= 0)
    constraints.append(E_variables[i] >= 0)
    constraints.append(H_storage_variables[i] >= 0)

constraints.append(H_storage_variables[0] == 0)  # Constraint the starting storage capacity of Hydrogen at 0

In [8]:
# Define the objective function
objective = cp.Minimize(sum(H_variables) * 10 + e_prices @ E_variables)

# Create the optimization problem
problem = cp.Problem(objective, constraints)

# Solve the problem
problem.solve()

# Print the results
print("Optimal value:", problem.value)
print(E_variables.value)
print(H_variables.value)
print(H_storage_variables.value)

# Solution.csv gets created in the current working directory and stores the values of the variables at optimal solution
with open('Solution.csv', 'w', newline='') as csvfile:
    my_writer = csv.writer(csvfile)
    # Write the header row
    my_writer.writerow(["Hour number", "Electricity Purchased", "Hydrogen Purchased"])
    # Write each element as a separate row in the CSV file
    for hour, E, H in zip(hour_numbers, E_variables.value, H_variables.value):
        my_writer.writerow([hour + 1, E, H])

# To determine which iteration of i is an active constraint.
for i in range(0, num_of_hours):
    slack = (H_variables.value[i] + H_storage_variables.value[i]) - h_demand[i]
    print("The slack for iteration %d is: %5.2f" % (i+1, slack))



Optimal value: 44408.999228645625
[2.10878447e+01 4.27762213e+01 7.34000598e+00 2.02858825e+01
 3.09926091e+01 1.69964765e+01 4.41808837e+01 2.22136158e+01
 2.81630114e+01 4.15588682e+01 3.67365693e+01 9.34150158e+00
 2.06732692e+01 3.00668358e+00 1.68640872e+01 5.38656312e-01
 1.79058221e+01 2.91369951e+01 3.93551675e+01 2.49420909e+01
 4.65405122e+01 2.06102666e+01 2.07386194e+01 1.77785947e+01
 3.28004136e+01 7.39280133e+00 5.75977724e+00 2.25220177e+00
 3.87191212e+01 4.07742902e+01 3.97506207e+01 1.53183819e+01
 2.20885707e+01 1.12960833e+01 1.19493677e-15 2.36626448e+01
 4.14411014e+01 2.64116116e+01 2.89406845e+01 4.89103663e+01
 3.41867428e+01 1.11272527e+00 5.43964058e+00 8.95619006e+00
 4.82263429e+01 9.48454447e+00 1.30544534e+01 2.81354571e+01
 4.87354252e+01 8.56874071e+00 2.19173070e+01 1.78206711e+01
 4.66583015e+01 2.75009939e+01 3.09974592e+01 2.21231141e+01
 3.61492266e+01 2.93487065e+01 1.14508133e+01 4.78012877e+01
 4.05190328e+01 3.18938808e+01 4.04915039e+01 3.671

In [9]:
# Loading datasets from CSV 
dataset = np.genfromtxt("6669_Project_Electricity_Demand.csv", delimiter=',')
e_demand = dataset[1:25, 1]
dataset = np.genfromtxt("6669_Project_Electricity_Prices.csv", delimiter=',')
e_prices = dataset[1:25, 2]
dataset = np.genfromtxt("6669_Project_Hydrogen_Demand.csv", delimiter=',')
h_demand = dataset[1:25, 1]
dataset = np.genfromtxt("6669_Project_Solar_Forecast.csv", delimiter=',')
solar = dataset[1:25, 1]

In [10]:
# Defining the number of hours/time range
# num_of_hours = 72
twenty_four_hours = 24
hour_numbers = list(range(0, twenty_four_hours))

# Defining decision variables
H_variables = cp.Variable(24)
E_variables = cp.Variable(24)
H_storage_variables = cp.Variable(25)  
# 73 to allow for the storage left at the end of the 72 hours

# Defining the new demand after Solar has been fully utilised and setting to zero of the demand is less than
# available solar energy.
e_new_demand = []
for i in range(0, twenty_four_hours):
    e_new_demand.append(e_demand[i] - 2 * solar[i])
    if e_new_demand[i] < 0:
        e_new_demand[i] = 0

# Create constraints using a for loop
constraints = []
for i in range(0, twenty_four_hours):
    constraints.append(H_variables[i] + H_storage_variables[i] >= h_demand[i])
    constraints.append(E_variables[i] == e_new_demand[i])
    constraints.append(H_storage_variables[i] - (h_demand[i] - H_variables[i]) - H_storage_variables[i + 1] == 0)

    # Sign Constraints
    constraints.append(H_variables[i] >= 0)
    constraints.append(E_variables[i] >= 0)
    constraints.append(H_storage_variables[i] >= 0)

constraints.append(H_storage_variables[0] == 0)  
# Constraint the starting storage capacity of Hydrogen at 0

# Define the objective function
objective = cp.Minimize(sum(H_variables) * 10 + e_prices @ E_variables)

# Create the optimization problem
problem = cp.Problem(objective, constraints)

# Solve the problem
problem.solve()

# Print the results
print("Optimal value:", problem.value)
# print(H_variables.value)
# print(E_variables.value)
# print(H_storage_variables.value)

# Solution.csv gets created in the current working directory and stores the values of the variables at optimal solution
with open('Solution_24_hours.csv', 'w', newline='') as csvfile:
    my_writer = csv.writer(csvfile)
    # Write the header row
    my_writer.writerow(["Hour number", "Electricity Purchased", "Hydrogen Purchased"])
    # Write each element as a separate row in the CSV file
    for hour, E, H in zip(hour_numbers, E_variables.value, H_variables.value):
        my_writer.writerow([hour + 1, E, H])

Optimal value: 13929.950886922863
